### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd


In [2]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 2048 
# dtype = ( None )
# load_in_4bit = False 


# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="./lora/lora_16bit_merged/",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [3]:
# # I highly do NOT suggest - use Unsloth if possible
# from peft import AutoPeftModelForCausalLM
# from transformers import AutoTokenizer
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#     load_in_4bit = False,
# )
# tokenizer = AutoTokenizer.from_pretrained("lora_model")

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define parameters
model_name = "./lora/lora_16bit_merged_2/"
max_seq_length = 2048  
torch_dtype = torch.float16  # Adjust dtype based on your model precision
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    device_map="auto"  # Automatically assigns model to GPU if available
)

# Ensure model is in evaluation mode for inference
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-0

In [5]:
from transformers import AutoProcessor, AutoModel
model_sl = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")
processor_sl = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")

In [6]:
import torch
from PIL import Image
import cairosvg
import os

def svgMetric(prompt, svg):
    try:
        # Convert SVG to PNG
        cairosvg.svg2png(svg, write_to="./tmp/temp.png")
        
        # Open and process the image
        image = Image.open('./tmp/temp.png').convert("RGB")
        texts = ["SVG illustration of " + prompt]
        inputs = processor_sl(text=texts, images=image, padding="max_length", return_tensors="pt")
        
        # Inference without gradient tracking
        with torch.no_grad():
            outputs = model_sl(**inputs)
        
        logits_per_image = outputs.logits_per_image
        probs = torch.sigmoid(logits_per_image)
        
        # Clean up temporary PNG file
        #os.remove('./tmp/temp.png')
        
        return probs[0][0].item()

    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [15]:
def get_response(text):
    # Define the Alpaca prompt template
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

    ### Instruction:
    Please write an SVG code for the given topic?

    ### Input:
    {}

    ### Response:
    """

    # Format the input text properly for the prompt
    formatted_input = alpaca_prompt.format(text)

    # Tokenize the formatted input
    inputs = tokenizer([formatted_input], return_tensors="pt").to("cuda")

    # Generate the response from the model
    outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True)

    # Decode and return the generated response
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]


In [16]:
import pandas as pd
df=pd.read_csv('svg_score_test.csv')
df=df[df['score'] > 0.6]
df=df[['topic','svg_code']]

In [17]:
from tqdm import tqdm
tqdm.pandas()
df['base_response'] = df['topic'].progress_apply(lambda x: get_response(x))

100%|███████████████████████████████████████████| 74/74 [09:07<00:00,  7.40s/it]


In [18]:
default_svg= '<svg viewBox="0 0 256 256" width="256" height="256">  </svg>'

import re
def clean_and_extract_svgs(text):

    cleaned_svgs = []
   
    # Remove any text before the first <svg>
    text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)

    # Find all <svg> tags (including nested)
    svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)

    if svg_blocks:
        # If there are multiple <svg> blocks, we pick the innermost one (which is the last in the list)
        tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
        if (len(tmp)) > 1:
            tmp2=svg_blocks[-1].split('<svg')
            cleaned_svgs.append('<svg '+tmp2[-1])
        else:
            cleaned_svgs.append(svg_blocks[-1])
            
    else:
        # Handle incomplete SVGs
        if "<svg" in text and "</svg>" not in text:
            # Remove any text before <svg> (again, in case it was incomplete)
            text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
            # Ensure it ends with a proper closing </svg>
            text += "</svg>"
            cleaned_svgs.append(text)
        else:
            cleaned_svgs.append(default_svg)  # No valid SVG found

           #removes the last inconplete element, like <incomp....</svg> 
    return cleaned_svgs[0].rsplit('\n', 1)[0] + '\n   </svg>'

#clean unconvertible svg with default svg
def svg_convertion_check(topic,base_svg_code):
    try:
        # Try to convert the SVG to PNG
        cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="./tmp/temp.png")
        return base_svg_code
        
    except Exception as e:
        # If conversion fails, return the default SVG
        print(f"Failed to convert {topic} due to {str(e)}. Returning default SVG.")
        return default_svg
        
# # Sample Input List
# svg_list = [
#     " ... <svg ..1..<comp>.... </svg>", 
#     "... <svg ..21..<comp1>.... </svg>.......<svg .22.<comp2>... </svg>...",
#     "... <svg ..3..<comp>.... <svg..3a..<comp>...</svg> ..<comp>..3f. </svg>...",
#     '<svg ..4..<comp>.... <svg..4a..<comp>...</svg> .4f.<incomppt....',
#     '....gsjkd..<svg...5..<comp>.....</svg>'
# ]
# # Process the SVG list
# result = clean_and_extract_svgs(svg_list)
# # Output Results
# for idx, svg in enumerate(result, 1):
#     print(f"SVG {idx}: {svg}\n")


In [19]:
df['base_svg_code'] = df['base_response'].progress_apply(lambda x: clean_and_extract_svgs(x))

100%|████████████████████████████████████████| 74/74 [00:00<00:00, 46111.80it/s]


In [20]:
# Using apply to process each row in the DataFrame
df['base_svg_code'] = df.progress_apply(lambda row: svg_convertion_check(row['topic'], row['base_svg_code']), axis=1)

100%|██████████████████████████████████████████| 74/74 [00:00<00:00, 396.56it/s]

Failed to convert  'Autumn forest with falling leaves', due to mismatched tag: line 30, column 7. Returning default SVG.
Failed to convert  'Dynamic fashion patterns with stripes', due to mismatched tag: line 46, column 5. Returning default SVG.
Failed to convert  'Desert oasis', due to duplicate attribute: line 28, column 29. Returning default SVG.
Failed to convert  'Concentric circles in a rainbow of colors.', due to mismatched tag: line 19, column 7. Returning default SVG.
Failed to convert  'Dynamic lines resembling ocean waves.', due to mismatched tag: line 12, column 7. Returning default SVG.
Failed to convert  'Golden sun rising over lush green fields.', due to mismatched tag: line 28, column 7. Returning default SVG.
Failed to convert  'Cozy fireplace in winter cabin', due to mismatched tag: line 27, column 7. Returning default SVG.


In [21]:
# Using apply to process each row in the DataFrame
df['base_score'] = df.progress_apply(lambda row: svgMetric(row['topic'], row['base_svg_code']), axis=1)

100%|███████████████████████████████████████████| 74/74 [01:51<00:00,  1.51s/it]


In [22]:
df['base_score'].mean()

np.float64(0.4482806710605009)